In [ ]:
from example_mortality_data import MALE_QX, FEMALE_QX

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

from matplotlib import cm

In [ ]:
"""Calculates an actuarially fair annuity from ages x to y.

If we pay an annuitant $1 each year its PV will be equal to 
probability of being alive and able to actually collect it.
Sum of those PVs is an Expected Present Value (EPV).

If a pool of infinite (so called "perfect pool") annuitants pays us $1 and we pay them back 1/EPV ->
In the end we should have $0 left on balance.

In case of non-zero inflation subtract the inflation rate from the interest rate.
The result will be an initial payment which then can be inflation-indexed.

Assumes subjective discount rate (SDR) == interest rate.

Annuity-due style!! Meaning the first payment gets handed out immediately.
"""

def calculate_and_plot_annuity(start_age, end_age, gender, interest_rate):
    # Normalize input and assign the corresponding mortality data
    qx_table = FEMALE_QX if gender.lower() == 'female' else MALE_QX
    
    # Ensure we don't index out of bounds if the user input exceeds the table length
    max_age = len(qx_table) - 1
    actual_end_age = min(end_age, max_age)
    
    # Initialize state: Annuity-due assumes the subject is alive at start_age (prob = 1.0)
    current_survival_prob = 1.0
    total_present_value = 0.0
    
    ages = []
    pv_values = []

    # Iterate through each year of the projected annuity period
    for age in range(start_age, actual_end_age + 1):
        # Calculate time elapsed since the start of the contract
        t = age - start_age
        
        # Determine the time-value of money using continuous discounting
        # Formula: PV = e^(-rt)
        discount_factor = math.exp(-interest_rate * t)
        
        # Calculate the Expected Present Value (EPV) for this specific year's $1 payment
        # This is the PV discounted for both interest and the probability of being alive
        pv_of_payment = discount_factor * current_survival_prob
        
        # Accumulate the EPV to find the total capital required to fund the annuity
        ages.append(age)
        pv_values.append(pv_of_payment)
        total_present_value += pv_of_payment
        
        # Update the cumulative survival probability for the next iteration (age + 1)
        # We multiply the current chance by (1 - probability of dying this year)
        if age < max_age:
            current_survival_prob *= (1 - qx_table[age])

    # Safety check
    if total_present_value <= 0:
        print("Error: Check for start_age < end_age or end_age out of bounds.")
        return 
    
    # The fair payout is multiplicative inverse of the total EPV. 
    fair_payout_rate = 1.0 / total_present_value
    
    # --- Visualization ---
    plt.style.use('dark_background')
    plt.figure(figsize=(10, 6))
    plt.axhline(0, color='grey', linewidth=1)
    
    # Plot the curve of 'Survival-Weighted PV' to show how the value of the 
    # $1 payment decays over time due to both interest and mortality risk.
    plt.plot(ages, pv_values, label='Survival-Weighted PV', color='white', linewidth=2)
    plt.fill_between(ages, pv_values, color='white', alpha=0.2) 
    
    plt.title(f'Actuarially Fair Annuity Payout for $1 paid (Start: {start_age}, End: {end_age}, r: {interest_rate*100}%)')
    plt.xlabel('Age')
    plt.ylabel('Present Value of $1 Payout')
    plt.grid(True, linestyle='--', alpha=0.15)
    
    # Overlay statistical summary on the chart area
    info_text = ("Total PV of \$1 perpetuity: \$" + f"{total_present_value:.2f}\n" + #\$ needed so $...$ cursive doesn't trigger
                 f"Fair Annual Payout: \${fair_payout_rate:.4f}")
    plt.gca().text(0.06, 0.07, info_text, transform=plt.gca().transAxes, 
                    fontsize=10, verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    plt.legend()
    plt.show()

    print("Total PV of $1 annuity: $" + f"{total_present_value}")
    print(f"Fair Annual Payout: ${fair_payout_rate}")

# --- Execution Parameters ---
# These are the variables a user or a calling script would modify
start_age = 60
end_age = 100
interest_rate = 0.03
gender = 'male'

if __name__ == "__main__":
    calculate_and_plot_annuity(start_age, end_age, gender, interest_rate)

In [ ]:
"""
This script performs a Monte Carlo stress-test on an actuarial pool.
By simulating thousands of 'parallel universes'
with random death events, it measures how survival variance (stochastic risk)
impacts the total fund balance over time.

Annuity-due style!!
Positive mean windfall might appear due to upward compounding bias.
"""

def run_annuity_simulation(
    n_sims=1000_000,
    initial_pop=1_000, 
    initial_contribution=1.0, 
    annual_payout=0.06519933571748932, 
    start_age=60, 
    end_age=100,
    interest_rate=0.03
):
    print(f"--- Running {n_sims} Simulations: {initial_pop} Retirees per pool ---")
    
    # Initialize the starting state for all simulation instances using NumPy vectors
    # This creates a baseline balance and population count for every 'parallel universe' in our simulation
    start_balance = initial_pop * initial_contribution
    pool_balances = np.full(n_sims, start_balance, dtype=float)
    populations = np.full(n_sims, initial_pop, dtype=int)
    
    ages = np.arange(start_age, end_age + 1)
    
    # Prepare 2D matrices to track how population and cash levels evolve at every age
    # Rows represent chronological steps; columns represent individual simulation runs
    pop_history = np.zeros((len(ages), n_sims))
    pool_history = np.zeros((len(ages), n_sims))

    for idx, age in enumerate(ages):
        # Capture the snapshot of all pools at the current age before any financial or mortality changes
        pop_history[idx, :] = populations
        pool_history[idx, :] = pool_balances
        
        # Execute the annual payout: subtract the total owed to all living members from the pool's capital
        total_payouts = populations * annual_payout
        pool_balances -= total_payouts
        
        # Apply investment growth to the remaining capital using continuous compounding
        # Logic ensures we only grow positive balances; we don't 'accrue interest' on debt/empty pools
        pool_balances = np.where(
            pool_balances > 0, 
            pool_balances * np.exp(interest_rate), 
            pool_balances
        )
        
        # Model the stochastic (random) nature of death using a Binomial distribution
        # For each pool, we 'flip a coin' for every member based on the specific mortality rate for that age
        if age < len(MALE_QX):
            mortality_rate = MALE_QX[age]
            deaths = np.random.binomial(populations, mortality_rate)
            populations -= deaths

    # Analyze the variance between the many simulation outcomes
    # The uncertainty ratio tells us how far the final balance drifted from the original deposit
    uncertainty_ratios = pool_balances / start_balance
    mean_end_balance = np.mean(pool_balances)
    volatility = np.std(uncertainty_ratios)

    print("-" * 65)
    print(f"Mean End Pool Balance: ${mean_end_balance:,.5f}")
    print(f"Start Pool Balance: ${start_balance:,.1f}")
    print(f"Mean Windfall/Start Balance: {np.mean(uncertainty_ratios):.3%}")
    
    print(f"\n1 std dev of end balance / start balance: {volatility * 1:.2%}")
    print(f"2 std dev of end balance / start balance: {volatility * 2:.2%}")
    print(f"3 std dev of end balance / start balance: {volatility * 3:.2%}")
    print("-" * 65)

    # Compress the massive simulation data into mean and standard deviation trends for clean visualization
    pop_mean = np.mean(pop_history, axis=1)
    pop_std = np.std(pop_history, axis=1)
    
    pool_mean = np.mean(pool_history, axis=1)
    pool_std = np.std(pool_history, axis=1)

    # --- Visualization Suite ---
    plt.style.use('dark_background')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Display the attrition of the retiree population over the simulated timeframe
    # The shaded area represents the 'luck of the draw' variance in survival across different pools
    ax1.axhline(0, color='grey', linewidth=1)
    ax1.plot(ages, pop_mean, color='white', linestyle='-', linewidth=1, label='Mean Population')
    ax1.fill_between(ages, pop_mean - 2*pop_std, pop_mean + 3*pop_std, 
                     color='white', alpha=0.2, label='±3 Std Dev')
    ax1.set_title('Surviving Retiree Population over Time', fontsize=14)
    ax1.set_xlabel('Age', fontsize=12)
    ax1.set_ylabel('Number of Retirees', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.15)

    # Display the financial health of the pool over time
    # Ideally, the mean balance should stay near zero in a perfectly 'fair' actuarial setup
    ax2.axhline(0, color='grey', linewidth=1)
    ax2.plot(ages, pool_mean, color='white', lw=1, label='Mean Balance')
    ax2.fill_between(ages, pool_mean - 2*pool_std, pool_mean + 3*pool_std, 
                     color='white', alpha=0.2, label='±3 Std Dev')
    ax2.set_title(f"Pool Balance over Time (r = {interest_rate*100:.1f}%)")
    ax2.set_xlabel("Age")
    ax2.set_ylabel("Balance ($)")
    ax2.legend()
    ax2.grid(True, alpha=0.15)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_annuity_simulation()

In [ ]:
"""
This script computes and visualizes the sensitivity of annuity payout rates across a broad 
spectrum of lifespans. By calculating the 'Fair Payout' for every possible combination 
of entry and exit ages, it generates a 3D surface (the 'payout landscape') that shows how 
drastically costs scale as the coverage window narrows or expands.

Annuity-due style!!
"""

def calculate_fair_payout(start_age, end_age, interest_rate=0.03):
    # Prevent illogical calculations where the contract ends before it starts
    if start_age > end_age or start_age >= len(MALE_QX):
        return np.nan 
    
    current_survival_prob = 1.0
    total_present_value = 0.0
    
    # Standard discrete summation of survival-weighted discount factors (PV)
    for age in range(start_age, end_age + 1):
        t = age - start_age
        # Compute the value of $1 today vs $1 at time 't' using continuous interest
        discount_factor = math.exp(-interest_rate * t)
        
        # Determine the expected cost of this specific year's obligation
        pv_of_payment = discount_factor * current_survival_prob
        total_present_value += pv_of_payment
        
        # Increment the probability of survival to the next year
        # We assume 100% mortality if the calculation exceeds the provided data table
        qx = MALE_QX[age] if age < len(MALE_QX) else 1.0
        current_survival_prob *= (1 - qx)
            
    # Return the reciprocal of the total Expected Present Value (EPV)
    return 1.0 / total_present_value if total_present_value > 0 else np.nan

if __name__ == "__main__":
    print("Generating 3D Payout Surface...")
    # --- Coordinate System Setup ---
    # Create a range of ages to test as both entry points (x) and exit points (y)
    start_ages = np.arange(20, 120, 1)
    end_ages = np.arange(20, 120, 1)

    # Generate a 2D coordinate grid representing all combinations of start and end ages
    start_age_grid, end_age_grid = np.meshgrid(start_ages, end_ages)

    v_get_payout = np.vectorize(calculate_fair_payout)

    interest_rate = 0.03

    payout_grid = v_get_payout(start_age_grid, end_age_grid, interest_rate)

    # --- 3D Visualization Logic ---
    plt.style.use('dark_background')
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Mask the 'impossible' half of the matrix (where start age is older than end age)
    # This prevents the plot from rendering artifacts in non-logical data regions
    payout_grid = np.ma.masked_where(start_age_grid > end_age_grid, payout_grid)

    surf = ax.plot_surface(start_age_grid, end_age_grid, payout_grid, cmap='plasma', edgecolor='navy', linewidth=0.5)

    # Configure chart metadata and axis descriptions
    ax.set_title(f'Actuarially Fair Annual Payouts (For $1 entry, annuity-due)\nRate: {interest_rate*100}%', pad=20)
    ax.set_xlabel('Starting Age (x)')
    ax.set_ylabel('Ending Age (y)')
    ax.set_zlabel('Annual Payout ($)')

    # Set the initial camera angle for the best perspective on the surface curvature
    ax.view_init(elev=30, azim=160)
    fig.colorbar(surf, shrink=0.5, aspect=10, label='Payout Amount')

    plt.tight_layout()
    plt.show()

In [ ]:
"""
This script calculates actuarially "fair" payouts based on mortality data and interest rates,
then simulates n life-expectancy scenarios to map how much the final pool balance varies across different age cohorts.
"""

def calculate_fair_payout(start_age, end_age, qx_table, interest_rate=0.03):
    # Establish the bounds of the simulation based on available mortality data
    max_age = len(qx_table) - 1
    actual_end_age = int(min(end_age, max_age))
    
    current_survival_prob = 1.0
    total_present_value = 0.0
    
    # Iterate through the lifespan to calculate the Expected Present Value (EPV)
    for age in range(int(start_age), actual_end_age + 1):
        # Calculate the discounted value of 1 unit paid at the start of each year (Annuity Due)
        discount_factor = np.exp(-interest_rate * (age - start_age))
        total_present_value += current_survival_prob * discount_factor
        
        # Update the probability of the individual surviving to the next year
        if age <= max_age:
            current_survival_prob *= (1 - qx_table[age])
            
    # Determine the constant annual payout that results in a zero net present value
    return 1.0 / total_present_value if total_present_value > 0 else 0

# --- Stochastic Simulation Engine ---

def calculate_volatility_vectorized(start_age, end_age, qx_table, payout, n_sims=500, initial_pop=1000, interest_rate=0.0):
    max_age = len(qx_table) - 1
    actual_end_age = int(min(end_age, max_age))
    
    # Initialize parallel simulation arrays for pool capital and survivor counts
    start_balance = initial_pop * 1.0
    pool_balances = np.full(n_sims, start_balance, dtype=float)
    populations = np.full(n_sims, initial_pop, dtype=int)
    
    # Step through each year of the annuity term
    for age in range(int(start_age), actual_end_age + 1):
        # Subtract the total annual liability from the pool for all living members
        pool_balances -= populations * payout
        
        # Grow the remaining capital using continuous interest compounding
        pool_balances = np.where(
            pool_balances > 0, 
            pool_balances * np.exp(interest_rate), 
            pool_balances
        )
        
        # Apply random mortality shocks using a Binomial distribution for each simulation trial
        if age <= max_age:
            mortality_rate = qx_table[age]
            deaths = np.random.binomial(populations, mortality_rate)
            populations -= deaths

    # Calculate the standard deviation of the outcome to measure the pool's financial risk
    return np.std(pool_balances / start_balance)

# --- Execution and Visualization ---

def plot_annuity_volatility_surface():
    # Define model hyperparameters and demographic data source
    qx_table = MALE_QX
    n_sims = 10_000
    initial_pop = 1_000
    interest_rate = 0.03
    
    # Create a 2D coordinate system representing start vs. end age variables
    ages = np.arange(60, 120, 1)
    start_age_grid, end_age_grid = np.meshgrid(ages, ages)
    volatility_grid = np.zeros_like(start_age_grid, dtype=float)

    print(f"Generating 3D Payout Surface...")
    for i in range(start_age_grid.shape[0]):
        for j in range(start_age_grid.shape[1]):
            start_age, end_age = start_age_grid[i, j], end_age_grid[i, j]
            if start_age > end_age:
                volatility_grid[i, j] = np.nan
            else:
                payout = calculate_fair_payout(start_age, end_age, qx_table, interest_rate)
                volatility_grid[i, j] = calculate_volatility_vectorized(
                    start_age, end_age, qx_table, payout,
                    n_sims=n_sims, initial_pop=initial_pop,
                    interest_rate=interest_rate
                )

    # Configure 3D plotting aesthetics
    plt.style.use('dark_background')
    sigmas = [1, 2, 3]
    
    # Generate separate plots for 1, 2, and 3 standard deviations (confidence intervals)
    for s in sigmas:
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Scale the base volatility by the current sigma level
        scaled_volatility_grid = volatility_grid * s
        
        surf = ax.plot_surface(start_age_grid, end_age_grid, scaled_volatility_grid, cmap=cm.plasma, edgecolor='navy', linewidth=0.6)
        
        # Format axes labels and plot titles dynamically
        if s == 1:
            ax.set_title(f'{s} Std Dev of Annuity Pool End Balance\n'
                         f'(Population={initial_pop}, Interest Rate={interest_rate})', fontsize=14)
            ax.set_zlabel(f'Std Dev ({s} Sigma)')
        else:
            ax.set_title(f'{s} Std Devs of Annuity Pool End Balance\n'
                         f'(Population={initial_pop}, Interest Rate={interest_rate})', fontsize=14)
            ax.set_zlabel(f'Std Dev ({s} Sigmas)')
            
        ax.set_xlabel('Starting Age (Years)')
        ax.set_ylabel('Ending Age (Years)')
        
        # Finalize layout and display the color scale bar
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
        ax.view_init(elev=30, azim=-30)
        
        plt.tight_layout()
        plt.show()

# --- Entry Point ---

if __name__ == "__main__":
    plot_annuity_volatility_surface()